# Migration des Données Médicales vers MongoDB

## Objectif
Ce notebook a pour but de démontrer les étapes nécessaires pour migrer un fichier de données médicales au format CSV vers une base de données MongoDB, avec les étapes suivantes :
1. Chargement et validation des données.
2. Nettoyage et export des données.
3. Insertion des données dans MongoDB.
4. Vérification des documents insérés dans la base de données.

## Fichiers requis
- **healthcare_dataset.csv** : Fichier source contenant les données médicales. Ce fichier doit être placé dans le répertoire `data/raw/`.

## Résultat attendu
Les données nettoyées seront enregistrées dans `data/processed/healthcare_data_cleaned.csv` et insérées dans une base MongoDB locale.


In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd  # Pour la manipulation des données
import os  # Pour interagir avec le système de fichiers
from loguru import logger  # Pour gérer les logs
from pymongo import MongoClient  # Pour se connecter à MongoDB
from pathlib import Path  # Pour gérer les chemins de fichiers et répertoires
import sys  # Pour modifier le chemin d'import des modules
import kagglehub  # Pour télécharger les datasets Kaggle
from skimpy import skim  # Résumés statistiques enrichis pour les DataFrames
import warnings   # Suppression des avertissements inutiles

# --- Configuration des Affichages pandas ---
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes d'un DataFrame
pd.set_option('display.max_rows', 100)      # Affiche jusqu'à 100 lignes dans un DataFrame
pd.set_option('display.width', 400)         # Évite les retours à la ligne inutiles

# --- Suppression des Avertissements Inutiles ---
warnings.filterwarnings("ignore")

# Configuration des logs
# Les logs sont enregistrés dans un fichier avec une rotation automatique pour limiter la taille
logger.add("../logs/migration.log", rotation="500 MB")
logger.info("Démarrage du notebook")

# Définir le répertoire de base pour un notebook ou un script
# Si on est dans un notebook, on remonte au répertoire parent
NOTEBOOK_PATH = Path(os.getcwd())
if NOTEBOOK_PATH.name == "notebooks":
    BASE_DIR = NOTEBOOK_PATH.parent
else:
    BASE_DIR = NOTEBOOK_PATH

# Ajouter le chemin du répertoire 'src' au sys.path
sys.path.append(str(BASE_DIR / "src"))

# Définir les chemins des répertoires à partir du répertoire de base
RAW_DIR = BASE_DIR / "data/raw"  # Répertoire pour les fichiers bruts
PROCESSED_DIR = BASE_DIR / "data/processed"  # Répertoire pour les fichiers nettoyés
SCRIPTS_DIR = BASE_DIR / "scripts"  # Répertoire des scripts

# --- Fonction pour configurer et loguer les répertoires ---
def setup_and_log_directories(base_dir, raw_dir, processed_dir, scripts_dir):
    """
    Configure les répertoires pour les fichiers bruts, nettoyés, et les scripts.
    Crée les répertoires si nécessaire et logue leur statut en chemins relatifs avec des descriptions.

    Returns:
        dict: Chemins relatifs et absolus des répertoires.
    """
    try:
        # Convertir les chemins en chemins relatifs
        base_dir_relative = base_dir.name  # Nom du répertoire racine
        raw_dir_relative = raw_dir.relative_to(base_dir)
        processed_dir_relative = processed_dir.relative_to(base_dir)
        scripts_dir_relative = scripts_dir.relative_to(base_dir)

        # Log du répertoire racine
        logger.info(f"Répertoire racine pour le projet défini : {base_dir_relative}")

        # Crée ou vérifie le répertoire des fichiers bruts
        raw_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Répertoire pour les fichiers bruts (RAW_DIR) configuré : {base_dir_relative}\\{raw_dir_relative}")

        # Crée ou vérifie le répertoire des fichiers nettoyés
        processed_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Répertoire pour les fichiers nettoyés (PROCESSED_DIR) configuré : {base_dir_relative}\\{processed_dir_relative}")

        # Crée ou vérifie le répertoire des scripts
        scripts_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Répertoire pour les scripts (SCRIPTS_DIR) configuré : {base_dir_relative}\\{scripts_dir_relative}")

        # Retourner les chemins relatifs et absolus
        return {
            "base_dir": base_dir,
            "raw_dir": {"absolute": raw_dir, "relative": raw_dir_relative},
            "processed_dir": {"absolute": processed_dir, "relative": processed_dir_relative},
            "scripts_dir": {"absolute": scripts_dir, "relative": scripts_dir_relative},
        }

    except Exception as e:
        # Log des erreurs en cas de problème
        logger.error(f"Erreur lors de la configuration des répertoires : {e}")
        raise

# Configuration des répertoires et récupération des chemins relatifs et absolus
directories = setup_and_log_directories(BASE_DIR, RAW_DIR, PROCESSED_DIR, SCRIPTS_DIR)

logger.info("Répertoires vérifiés et créés avec succès.")

: 

In [ ]:
# -------------------------------------------
# Téléchargement et Chargement des Données
# -------------------------------------------

# Téléchargement du dataset depuis Kaggle
try:
    # Télécharge le dataset en utilisant KaggleHub
    dataset_path = kagglehub.dataset_download("prasad22/healthcare-dataset")
    logger.info(f"Dataset téléchargé avec succès")
except Exception as e:
    # Gère les erreurs de téléchargement et les logue
    logger.error(f"Erreur lors du téléchargement du dataset : {e}")
    raise  # Arrête l'exécution en cas d'échec

# Étape 2 : Lister les fichiers disponibles dans le dossier téléchargé
try:
    # Liste tous les fichiers dans le répertoire du dataset
    files = os.listdir(dataset_path)
    logger.info(f"Fichiers disponibles dans le dataset : {files}")
except Exception as e:
    # Log en cas d'échec de la lecture du répertoire
    logger.error(f"Erreur lors de la lecture des fichiers dans le répertoire {dataset_path} : {e}")
    raise

# Étape 3 : Chargement d'un fichier spécifique
# Nom du fichier attendu
dataset_file = Path(dataset_path) / "healthcare_dataset.csv"
try:
    # Charge le fichier CSV en mémoire en tant que DataFrame pandas
    data = pd.read_csv(dataset_file)
    logger.info(f"Fichier '{dataset_file.name}' chargé avec succès. "
                f"Nombre de lignes : {data.shape[0]}. Nombre de colonnes : {data.shape[1]}.")
except FileNotFoundError:
    # Log en cas de fichier manquant
    logger.error(f"Le fichier spécifié '{dataset_file.name}' est introuvable dans le dataset.")
    raise
except Exception as e:
    # Log pour tout autre type d'erreur
    logger.error(f"Erreur lors du chargement du fichier '{dataset_file.name}' : {e}")
    raise


In [ ]:
# 🚀 Exportation des DataFrames en CSV avec encodage UTF-8
logger.info("=" * 60)
logger.info("Début de l'exportation des DataFrames en fichiers CSV")
logger.info("-" * 10)

def export_dataframes_to_csv(dataframes, output_directory, base_dir):
    """
    Exporte chaque DataFrame en fichier CSV dans le répertoire spécifié avec encodage UTF-8.

    Args:
        dataframes (dict): Dictionnaire de DataFrames à exporter.
        output_directory (Path): Répertoire où sauvegarder les fichiers CSV.
        base_dir (Path): Répertoire racine pour générer des chemins relatifs dans les logs.
    """
    if not isinstance(output_directory, Path):
        logger.error(f"Le chemin '{output_directory}' n'est pas une instance de Path.")
        return

    if not output_directory.exists():
        logger.warning(f"Le répertoire '{output_directory}' n'existe pas. Création du répertoire.")
        output_directory.mkdir(parents=True, exist_ok=True)

    for name, df in dataframes.items():
        if not isinstance(df, pd.DataFrame):
            logger.error(f"'{name}' n'est pas un DataFrame valide. Exportation ignorée.")
            continue

        export_path = output_directory / f"{name}.csv"
        try:
            # Exporter le DataFrame en CSV
            df.to_csv(export_path, index=False, encoding='utf-8')

            # Convertir le chemin absolu en chemin relatif pour les logs
            relative_path = export_path.relative_to(base_dir)
            logger.info(f"✅ DataFrame '{name}' exporté avec succès : {relative_path}")
        except Exception as e:
            logger.error(f"❌ Erreur lors de l'exportation du DataFrame '{name}': {e}")

 
healthcare_dataset = {"healthcare_data": data}
export_dataframes_to_csv(healthcare_dataset, directories["raw_dir"]["absolute"], BASE_DIR)

logger.info("-" * 10)
logger.info("Fin de l'exportation des DataFrames en fichiers CSV")
logger.info("=" * 60)

In [ ]:
# Aperçu des données
# Afficher les 5 premières lignes pour un aperçu rapide du contenu
logger.info(f"Aperçu des données : \n{data.head()}\n")

# Résumé statistique avec skim
# Fournit une vue d'ensemble rapide des colonnes, types, valeurs manquantes, etc.
logger.info(f"Résumé pour le dataset 'healthcare_dataset':")
skim(data)  # Assurez-vous que skimpy est installé : pip install skimpy

# Séparateur visuel dans les logs
logger.info(f"{'=' * 60}")

# Types des colonnes
# Affiche les types des colonnes pour identifier d'éventuels problèmes
logger.info(f"📊 Types des colonnes pour le DataFrame 'healthcare'")
logger.info(f"{'-' * 60}")

# Boucle pour afficher les détails de chaque colonne
# Cela inclut le type de données pandas et les types Python uniques rencontrés
for col in data.columns:
    col_type = data[col].dtype  # Type de colonne selon pandas (int64, object, etc.)
    unique_types = data[col].apply(lambda x: type(x).__name__).unique()  # Types Python rencontrés
    logger.info(f"🔹 {col} : {col_type} | Valeurs : {', '.join(unique_types)}")

# Séparateur final
logger.info(f"{'=' * 60}\n")


In [ ]:
# Identification des doublons
data['is_duplicate'] = data.duplicated(keep=False)

# Filtrer les lignes marquées comme doublons
duplicates = data[data['is_duplicate']].copy()

# Ajouter une colonne 'is_primary' pour identifier la première occurrence
duplicates['is_primary'] = ~duplicates.duplicated(keep='first')

# Normaliser les colonnes avant le traitement
columns_to_normalize = [col for col in data.columns if col not in ['is_duplicate', 'is_primary']]
for col in columns_to_normalize:
    if data[col].dtype == 'object':  # Normaliser uniquement les colonnes de type texte
        duplicates[col] = duplicates[col].str.strip().str.lower()  # Supprimer les espaces et uniformiser les majuscules

# Ajouter un identifiant temporaire unique pour chaque ligne
duplicates = duplicates.reset_index(drop=False).rename(columns={'index': 'temp_id'})

# Séparer les lignes principales et les doublons
primary_rows = duplicates[duplicates['is_primary']].copy()
duplicate_rows = duplicates[~duplicates['is_primary']].copy()

# Ajouter un identifiant unique pour les groupes de doublons
primary_rows['group_id'] = range(1, len(primary_rows) + 1)

# Associer les doublons à leurs lignes principales
duplicates_with_group = duplicate_rows.merge(
    primary_rows[columns_to_normalize + ['group_id']], 
    how='left', 
    on=columns_to_normalize, 
    suffixes=('', '_primary')
)

# Vérification des colonnes générées après le merge
generated_columns = duplicates_with_group.columns
logger.info(f"Colonnes générées après le merge : {generated_columns}")

# Comparer chaque colonne entre les lignes principales et leurs doublons
diff_columns = []  # Initialisation de la liste des colonnes de différences
for col in columns_to_normalize:
    primary_col = f"{col}_primary"
    if primary_col in duplicates_with_group.columns:
        diff_col = f"{col}_diff"
        duplicates_with_group[diff_col] = duplicates_with_group[col] != duplicates_with_group[primary_col]
        diff_columns.append(diff_col)  # Ajouter à la liste des colonnes de différences
    else:
        logger.warning(f"La colonne '{primary_col}' n'existe pas. Comparaison ignorée.")

# Ajouter la liste des groupes concernés par des doublons
if not duplicates_with_group.empty:  # Vérifie si le DataFrame n'est pas vide
    group_ids = duplicates_with_group['group_id'].unique()  # Extraire les identifiants des groupes
    logger.info(f"Groupes concernés par des doublons : {group_ids.tolist()}")
else:
    logger.info("Aucun groupe concerné par des doublons.")


# Résultat final pour analyse
logger.info(f"Groupes de doublons avec colonnes de différences : \n{duplicates_with_group.head()}")
logger.info(f"Colonnes avec des différences détectées : {diff_columns}")


In [ ]:
# Liste des groupes à analyser
groups_to_check = [81, 21, 279, 61, 211, 383, 112, 196, 231, 55, 309, 508, 172, 384, 387, 202, 450, 92, 481, 432, 223, 307]  # Remplacez par les identifiants des groupes que vous souhaitez analyser

# Filtrer les lignes principales et les doublons pour les groupes sélectionnés
groups_primary = primary_rows[primary_rows['group_id'].isin(groups_to_check)]
groups_duplicates = duplicates_with_group[duplicates_with_group['group_id'].isin(groups_to_check)]

# Combiner les lignes principales et les doublons pour analyse
groups_to_inspect = pd.concat([groups_primary, groups_duplicates])

# Trier les résultats par 'group_id' pour regrouper les lignes par groupe
groups_to_inspect = groups_to_inspect.sort_values(by=['group_id', 'is_primary'], ascending=[True, False])

# Afficher le contenu des groupes pour validation
logger.info(f"Contenu complet des groupes triés par 'group_id' : \n{groups_to_inspect}")


In [ ]:
# Identification des doublons
data['is_duplicate'] = data.duplicated(keep=False)

# Filtrer les lignes marquées comme doublons
duplicates = data[data['is_duplicate']].copy()

# Ajouter une colonne 'is_primary' pour identifier la première occurrence
duplicates['is_primary'] = ~duplicates.duplicated(keep='first')

# Rattacher la colonne 'is_primary' à l'ensemble du DataFrame
data['is_primary'] = data.index.isin(duplicates.index[duplicates['is_primary']])

# Garder uniquement les lignes principales et les lignes sans doublons
cleaned_data = data[data['is_primary'] | ~data['is_duplicate']].copy()

# Supprimer les colonnes ajoutées pour l’analyse des doublons
columns_to_remove = ['is_primary', 'is_duplicate', 'group_id']
cleaned_data.drop(columns=columns_to_remove, inplace=True, errors='ignore')

# Afficher le résultat final
logger.info(f"Nombre de lignes après suppression des doublons : {cleaned_data.shape[0]}")
logger.info(f"Aperçu des données nettoyées : \n{cleaned_data.head()}")


In [ ]:
# --- Analyse avant transformation ---

# Analyse initiale des colonnes et des types
logger.info("Analyse initiale des données :")
logger.info(f"Colonnes disponibles : {list(cleaned_data.columns)}")
logger.info(f"Types des colonnes avant transformation : \n{cleaned_data.dtypes}")

# Statistiques descriptives pour les colonnes de date et de texte
columns_to_analyze = ['Date of Admission', 'Discharge Date', 'Doctor', 'Hospital']
for col in columns_to_analyze:
    if col in cleaned_data.columns:
        logger.info(f"Statistiques pour '{col}': \n{cleaned_data[col].describe(include='all')}")

# Validation des colonnes catégoriques avant transformation
expected_gender = {'Male', 'Female'}
unexpected_gender = set(cleaned_data['Gender'].unique()) - expected_gender
logger.info(f"Valeurs initiales pour 'Gender' : {set(cleaned_data['Gender'].unique())}")
if unexpected_gender:
    logger.warning(f"Valeurs inattendues trouvées dans 'Gender' : {unexpected_gender}")

expected_blood_types = {'A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-'}
unexpected_blood_types = set(cleaned_data['Blood Type'].unique()) - expected_blood_types
logger.info(f"Valeurs initiales pour 'Blood Type' : {set(cleaned_data['Blood Type'].unique())}")
if unexpected_blood_types:
    logger.warning(f"Valeurs inattendues trouvées dans 'Blood Type' : {unexpected_blood_types}")


In [ ]:
# --- Transformation des données ---

# Transformation des colonnes de dates
columns_to_convert = ['Date of Admission', 'Discharge Date']
for col in columns_to_convert:
    if col in cleaned_data.columns:
        cleaned_data[col] = pd.to_datetime(cleaned_data[col], errors='coerce')
logger.info("Les colonnes de dates ont été converties en type datetime.")

# Normalisation des colonnes de texte
columns_to_normalize = ['Doctor', 'Hospital']
for col in columns_to_normalize:
    if col in cleaned_data.columns:
        cleaned_data[col] = cleaned_data[col].str.strip().str.title()
logger.info("Les colonnes de texte ont été normalisées (espaces supprimés, titres formatés).")


In [ ]:
# --- Analyse après transformation ---

# Analyse après transformation
logger.info("Analyse après transformation des données :")
logger.info(f"Types des colonnes après transformation : \n{cleaned_data.dtypes}")

# Comparaison des colonnes de dates
for col in columns_to_convert:
    if col in cleaned_data.columns:
        logger.info(f"Exemple de valeurs après conversion pour '{col}': \n{cleaned_data[col].head()}")

# Vérification des colonnes normalisées
for col in columns_to_normalize:
    if col in cleaned_data.columns:
        logger.info(f"Exemple de valeurs après normalisation pour '{col}': \n{cleaned_data[col].head()}")

In [ ]:
# 🚀 Exportation des DataFrames en CSV avec encodage UTF-8
logger.info("=" * 60)
logger.info("Début de l'exportation des DataFrames en fichiers CSV")
logger.info("-" * 10)

cleaned_dataframes = {"healthcare_data": cleaned_data}
export_dataframes_to_csv(cleaned_dataframes, directories["processed_dir"]["absolute"], BASE_DIR)

logger.info("-" * 10)
logger.info("Fin de l'exportation des DataFrames en fichiers CSV")
logger.info("=" * 60)


In [ ]:
# Ajouter SCRIPTS_DIR au chemin Python pour importer mongodb_crud
sys.path.append(str(directories["scripts_dir"]["absolute"]))

# Importer les fonctions du module mongodb_crud
from mongodb_crud import connect_to_mongodb, insert_records, read_records, update_records, delete_records

# Connexion à MongoDB
db = connect_to_mongodb()
collection = db["patients_data"]

# CREATE : Insérer toutes les données dans MongoDB
records = data.to_dict(orient="records")
insert_records(collection, records)

In [ ]:
# READ : Lire un échantillon des documents
logger.info("Lecture d'un échantillon des données insérées :")
sample_records = read_records(collection, {}, limit=5)
print("Échantillon des données :", sample_records)


In [ ]:
# READ : Filtrer par une condition spécifique
logger.info("Lecture des patients ayant un âge supérieur à 50 :")
age_filter = {"Age": {"$gt": 50}}
patients_over_50 = read_records(collection, age_filter, limit=5)
print("Patients de plus de 50 ans :", patients_over_50)

In [ ]:
# UPDATE : Mettre à jour les documents
logger.info("Mise à jour des documents pour les patients de plus de 50 ans.")
update_filter = {"Age": {"$gt": 50}}
update_query = {"$set": {"Status": "Senior"}}
update_count = update_records(collection, update_filter, update_query)
logger.info(f"Nombre de documents mis à jour : {update_count}")


In [ ]:
# READ : Vérifier les documents mis à jour
logger.info("Vérification des documents mis à jour :")
updated_records = read_records(collection, {"Status": "Senior"}, limit=5)
print("Documents mis à jour :", updated_records)

In [ ]:
# DELETE : Supprimer les documents ayant un statut "Senior"
logger.info("Suppression des documents ayant le statut 'Senior'.")
delete_filter = {"Status": "Senior"}
delete_count = delete_records(collection, delete_filter)
logger.info(f"Nombre de documents supprimés : {delete_count}")

# READ : Vérifier la suppression
logger.info("Vérification après suppression :")
remaining_records = read_records(collection, {}, limit=5)
print("Documents restants :", remaining_records)

# Détails des opérations CRUD

## CREATE :
- Insertion de toutes les données du fichier CSV dans MongoDB.

## READ :
- Lecture d’un échantillon de documents.
- Lecture de documents filtrés (patients de plus de 50 ans).

## UPDATE :
- Mise à jour d’un champ (`Status`) pour les patients de plus de 50 ans.

## DELETE :
- Suppression des documents ayant un champ `Status` égal à `Senior`.

## Validation :
- Chaque étape est suivie d’une vérification pour confirmer le résultat.
